In [40]:
import pandas as pd
import numpy as np
from datetime import date

df = pd.read_csv("HRDataset_v14.csv")  
reference_date = pd.Timestamp(date.today())

# les colomnes
leakage = [
    "DateofTermination", 
    "TermReason",          
    "EmploymentStatus",    
    "EmpStatusID",         
]
df.drop(columns=leakage, inplace=True)

identifiers = ["Employee_Name", "EmpID", "Department", "ManagerName", "Zip", "State"]
df.drop(columns=identifiers, inplace=True)

text_duplicates = [
    "Position",        
    "MaritalDesc",     
    "Sex", "PerformanceScore",
]
df.drop(columns=text_duplicates, inplace=True)

def parse_date(col):
    return pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")

def fix_future_dates(series):
    return series.where(series <= reference_date, series - pd.DateOffset(years=100))

dob_parsed = fix_future_dates(parse_date("DOB"))
df["Age"] = ((reference_date - dob_parsed).dt.days / 365.25).astype(int)


df.drop(columns=["DOB", "DateofHire", "LastPerformanceReview_Date"], inplace=True)

df["HispanicLatino"] = df["HispanicLatino"].str.strip().str.lower().map(
    {"yes": 1, "no": 0}
)

ohe_cols = ["CitizenDesc", "RaceDesc", "RecruitmentSource"]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

df.to_csv("hr_cleaned.csv", index=False)

C:\Users\othma\AppData\Local\Temp\ipykernel_32468\3323835041.py:28: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  return pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")
C:\Users\othma\AppData\Local\Temp\ipykernel_32468\3323835041.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(df[col], infer_datetime_format=True, errors="coerce")


In [41]:
df.head()

,MarriedID,MaritalStatusID,GenderID,DeptID,PerfScoreID,FromDiversityJobFairID,Salary,Termd,PositionID,HispanicLatino,...,RaceDesc_Two or more races,RaceDesc_White,RecruitmentSource_Diversity Job Fair,RecruitmentSource_Employee Referral,RecruitmentSource_Google Search,RecruitmentSource_Indeed,RecruitmentSource_LinkedIn,RecruitmentSource_On-line Web application,RecruitmentSource_Other,RecruitmentSource_Website
0,0,0,1,5,4,0,62506,0,19,0,...,False,True,False,False,False,False,True,False,False,False
1,1,1,1,3,3,0,104437,1,27,0,...,False,True,False,False,False,True,False,False,False,False
2,1,1,0,5,3,0,64955,1,20,0,...,False,True,False,False,False,False,True,False,False,False
3,1,1,0,5,3,0,64991,0,19,0,...,False,True,False,False,False,True,False,False,False,False
4,0,2,0,5,3,0,50825,1,19,0,...,False,True,False,False,True,False,False,False,False,False
